# 15. Optimización y selección de modelos finales

## 15.1. Objetivo

En los notebooks anteriores se han entrenado y comparado diferentes algoritmos de clasificación para predecir los códigos de calidad de las tres componentes de irradiancia: GHI, DNI y DHI.

El objetivo de este notebook es seleccionar los modelos con mejor rendimiento obtenidos previamente y optimizar sus hiperparámetros para obtener una configuración final para cada variable objetivo.

La optimización se realizará utilizando exclusivamente los datos correspondientes a 2024. Los datos de 2023 permanecerán completamente aislados durante esta fase y se utilizarán únicamente al final del proceso para realizar una evaluación interanual independiente de los modelos optimizados.

La métrica principal utilizada para seleccionar las configuraciones será Macro F1, debido al desequilibrio existente entre las clases y al interés en evaluar de forma equilibrada el rendimiento sobre códigos mayoritarios y minoritarios. Como métricas complementarias se utilizarán Balanced Accuracy y Weighted F1.

El flujo seguido será:

1. Recuperación de los mejores modelos obtenidos en los notebooks anteriores.
2. Definición de los espacios de hiperparámetros.
3. Optimización utilizando únicamente los datos de entrenamiento de 2024.
4. Selección de la mejor configuración para cada target.
5. Evaluación final sobre los datos independientes de 2023.
6. Comparación entre modelos originales y optimizados.
7. Persistencia de los modelos y resultados finales.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report
)

RANDOM_STATE = 42

TRAIN_YEAR = 2024
TEST_YEAR = 2023

TARGETS = [
    "codigo_ghi",
    "codigo_dni",
    "codigo_dhi"
]

PRIMARY_METRIC = "f1_macro"

DATA_PATH = Path(
    "../data/processed/dataset_solar_2023_2024_v3.parquet"
)

MODEL_OUTPUT_PATH = Path(
    "../models/final"
)

MODEL_OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

In [2]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [3]:
df = pd.read_parquet(DATA_PATH)

print(f"Dataset completo: {df.shape}")
print(
    f"Periodo: {df['fecha'].min()} "
    f"→ {df['fecha'].max()}"
)

Dataset completo: (1052640, 27)
Periodo: 2023-01-01 00:00:00 → 2024-12-31 23:59:00


In [4]:
train_df = (
    df[df["ano"] == TRAIN_YEAR]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    df[df["ano"] == TEST_YEAR]
    .copy()
    .reset_index(drop=True)
)

print(
    f"Train ({TRAIN_YEAR}): "
    f"{train_df.shape}"
)

print(
    f"Test ({TEST_YEAR}): "
    f"{test_df.shape}"
)

Train (2024): (527040, 27)
Test (2023): (525600, 27)


In [5]:
EXCLUDED_COLUMNS = [
    "fecha",
    "ano",
    "codigo_ghi",
    "codigo_dni",
    "codigo_dhi"
]

CANDIDATE_FEATURES = [
    column
    for column in df.columns
    if column not in EXCLUDED_COLUMNS
]

print(
    f"Variables candidatas disponibles: "
    f"{len(CANDIDATE_FEATURES)}"
)

CANDIDATE_FEATURES

Variables candidatas disponibles: 22


['mes_sin',
 'mes_cos',
 'dia',
 'hora_sin',
 'hora_cos',
 'minuto',
 'ghi',
 'dni',
 'dhi',
 'ghi_estimado',
 'irr_null',
 'error_balance',
 'error_balance_abs',
 'error_balance_rel',
 'elevacion_solar',
 'periodo_solar',
 'temperatura',
 'velocidad_viento',
 'humedad_relativa',
 'direccion_viento_sin',
 'direccion_viento_cos',
 'var_meteo_imp']

In [6]:
from src.database.postgresql_persistence import get_postgresql_connection

In [7]:
query_all_results = """
SELECT
    m.model_id,
    m.model_name,
    m.target,
    m.train_year,
    m.test_year,
    m.n_features,
    m.features,
    m.hyperparameters,
    r.f1_macro,
    r.balanced_accuracy,
    r.f1_weighted,
    r.created_at
FROM solar.models AS m
INNER JOIN solar.results AS r
    ON m.model_id = r.model_id
ORDER BY
    m.target,
    r.f1_macro DESC;
"""

conn = get_postgresql_connection()

try:
    all_results = pd.read_sql_query(
        query_all_results,
        conn
    )
finally:
    conn.close()

print(
    f"Número total de experimentos registrados: "
    f"{len(all_results)}"
)

all_results.head()

Número total de experimentos registrados: 18


C:\Users\zacar\AppData\Local\Temp\ipykernel_14532\3418399148.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  all_results = pd.read_sql_query(


,model_id,model_name,target,train_year,test_year,n_features,features,hyperparameters,f1_macro,balanced_accuracy,f1_weighted,created_at
0,3,HistGradientBoostingClassifier,codigo_dhi,2024,2023,22,"[mes_sin, mes_cos, dia, hora_sin, hora_cos, mi...","{'tol': 1e-07, 'loss': 'log_loss', 'scoring': ...",0.498600,0.572575,0.882477,2026-08-13 18:41:04.711730
1,21,HistGradientBoosting,codigo_dhi,2024,2023,18,"[mes_sin, mes_cos, dia, hora_sin, hora_cos, mi...","{'tol': 1e-07, 'loss': 'log_loss', 'scoring': ...",0.478681,0.555883,0.863185,2026-08-18 08:11:53.092781
2,24,MLP,codigo_dhi,2024,2023,14,"[mes_sin, mes_cos, dia, hora_sin, hora_cos, mi...","{'optimizer': 'Adam', 'activation': 'relu', 'b...",0.451350,0.515467,0.842825,2026-08-19 08:44:36.752629
3,18,RandomForest,codigo_dhi,2024,2023,14,"[mes_sin, mes_cos, dia, hora_sin, hora_cos, mi...","{'n_jobs': -1, 'verbose': 0, 'bootstrap': True...",0.450474,0.505040,0.847037,2026-08-18 08:11:42.893393
4,15,LogisticRegression,codigo_dhi,2024,2023,14,"[mes_sin, mes_cos, dia, hora_sin, hora_cos, mi...","{'C': 1.0, 'tol': 0.0001, 'dual': False, 'n_jo...",0.413080,0.560074,0.778568,2026-08-18 08:11:24.517424


In [8]:
results_summary = all_results[
    [
        "model_id",
        "target",
        "model_name",
        "n_features",
        "f1_macro",
        "balanced_accuracy",
        "f1_weighted"
    ]
].copy()

results_summary

,model_id,target,model_name,n_features,f1_macro,balanced_accuracy,f1_weighted
0,3,codigo_dhi,HistGradientBoostingClassifier,22,0.498600,0.572575,0.882477
1,21,codigo_dhi,HistGradientBoosting,18,0.478681,0.555883,0.863185
2,24,codigo_dhi,MLP,14,0.451350,0.515467,0.842825
3,18,codigo_dhi,RandomForest,14,0.450474,0.505040,0.847037
4,15,codigo_dhi,LogisticRegression,14,0.413080,0.560074,0.778568
5,12,codigo_dhi,DummyClassifier,0,0.313993,0.333333,0.838667
6,20,codigo_dni,HistGradientBoosting,18,0.534738,0.625648,0.940909
7,17,codigo_dni,RandomForest,14,0.489022,0.551082,0.927517
8,2,codigo_dni,HistGradientBoostingClassifier,22,0.486994,0.616805,0.932485
9,23,codigo_dni,MLP,14,0.462769,0.675691,0.904146


In [9]:
best_results = (
    all_results
    .sort_values(
        by=[
            "target",
            "f1_macro",
            "balanced_accuracy",
            "f1_weighted"
        ],
        ascending=[
            True,
            False,
            False,
            False
        ]
    )
    .drop_duplicates(
        subset="target",
        keep="first"
    )
    .reset_index(drop=True)
)

best_results[
    [
        "model_id",
        "target",
        "model_name",
        "n_features",
        "f1_macro",
        "balanced_accuracy",
        "f1_weighted"
    ]
]

,model_id,target,model_name,n_features,f1_macro,balanced_accuracy,f1_weighted
0,3,codigo_dhi,HistGradientBoostingClassifier,22,0.498600,0.572575,0.882477
1,20,codigo_dni,HistGradientBoosting,18,0.534738,0.625648,0.940909
2,22,codigo_ghi,MLP,14,0.743533,0.782470,0.883894


In [10]:
best_candidates = {
    "codigo_ghi": [
        "MLP",
        "LogisticRegression"
    ],
    "codigo_dni": [
        "HistGradientBoosting"
    ],
    "codigo_dhi": [
        "HistGradientBoostingClassifier"
    ]
}

In [11]:
candidates_to_optimize = pd.concat([
    
    # GHI: dos mejores candidatos
    (
        all_results[
            all_results["target"] == "codigo_ghi"
        ]
        .sort_values("f1_macro", ascending=False)
        .head(2)
    ),
    
    # DNI: mejor candidato
    (
        all_results[
            all_results["target"] == "codigo_dni"
        ]
        .sort_values("f1_macro", ascending=False)
        .head(1)
    ),
    
    # DHI: mejor candidato
    (
        all_results[
            all_results["target"] == "codigo_dhi"
        ]
        .sort_values("f1_macro", ascending=False)
        .head(1)
    )
    
]).reset_index(drop=True)

candidates_to_optimize[
    [
        "model_id",
        "target",
        "model_name",
        "n_features",
        "f1_macro",
        "balanced_accuracy",
        "f1_weighted"
    ]
]

,model_id,target,model_name,n_features,f1_macro,balanced_accuracy,f1_weighted
0,22,codigo_ghi,MLP,14,0.743533,0.782470,0.883894
1,13,codigo_ghi,LogisticRegression,14,0.646299,0.810603,0.790116
2,20,codigo_dni,HistGradientBoosting,18,0.534738,0.625648,0.940909
3,3,codigo_dhi,HistGradientBoostingClassifier,22,0.498600,0.572575,0.882477


In [12]:
train_df["mes_validation"] = train_df["fecha"].dt.month

monthly_target_distribution = {}

for target in TARGETS:
    monthly_distribution = pd.crosstab(
        train_df["mes_validation"],
        train_df[target],
        normalize="index"
    )

    monthly_target_distribution[target] = monthly_distribution

    print(f"\n{'=' * 70}")
    print(f"Distribución mensual de clases — {target}")
    print(f"{'=' * 70}")
    
    display(monthly_distribution.round(4))


Distribución mensual de clases — codigo_ghi


codigo_ghi,0,1
mes_validation,,
1,0.9979,0.0021
2,0.8465,0.1535
3,0.5476,0.4524
4,0.8638,0.1362
5,0.9026,0.0974
6,0.9429,0.0571
7,0.7765,0.2235
8,0.9800,0.0200
9,0.4329,0.5671



Distribución mensual de clases — codigo_dni


codigo_dni,0,1,2
mes_validation,,,
1,0.9959,0.0000,0.0041
2,0.8437,0.1552,0.0011
3,0.5471,0.4528,0.0002
4,0.8571,0.1385,0.0044
5,0.9004,0.0978,0.0019
6,0.9642,0.0337,0.0020
7,0.8052,0.1946,0.0002
8,0.9974,0.0015,0.0012
9,0.4329,0.5671,0.0000



Distribución mensual de clases — codigo_dhi


codigo_dhi,0,1,2
mes_validation,,,
1,0.9904,0.0096,0.0000
2,0.8415,0.1585,0.0000
3,0.5465,0.4524,0.0011
4,0.8539,0.1461,0.0000
5,0.7405,0.2595,0.0000
6,0.5669,0.4331,0.0000
7,0.7316,0.2649,0.0035
8,0.9080,0.0920,0.0000
9,0.4329,0.5671,0.0000


In [13]:
class_presence_by_month = []

for target in TARGETS:
    for month in sorted(train_df["mes_validation"].unique()):
        month_data = train_df.loc[
            train_df["mes_validation"] == month,
            target
        ]

        counts = month_data.value_counts()

        class_presence_by_month.append({
            "target": target,
            "mes_validation": month,
            "n_samples": len(month_data),
            "n_classes": month_data.nunique(),
            "classes": sorted(month_data.unique().tolist()),
            "min_class_count": counts.min()
        })

class_presence_by_month = pd.DataFrame(
    class_presence_by_month
)

class_presence_by_month

,target,mes_validation,n_samples,n_classes,classes,min_class_count
0,codigo_ghi,1,44640,2,"[0, 1]",94
1,codigo_ghi,2,41760,2,"[0, 1]",6409
2,codigo_ghi,3,44640,2,"[0, 1]",20194
3,codigo_ghi,4,43200,2,"[0, 1]",5885
4,codigo_ghi,5,44640,2,"[0, 1]",4350
5,codigo_ghi,6,43200,2,"[0, 1]",2468
6,codigo_ghi,7,44640,2,"[0, 1]",9976
7,codigo_ghi,8,44640,2,"[0, 1]",892
8,codigo_ghi,9,43200,2,"[0, 1]",18702
9,codigo_ghi,10,44640,2,"[0, 1]",1752


In [14]:
VALIDATION_MONTHS = [3, 8, 11]

optimization_train_df = (
    train_df[
        ~train_df["mes_validation"].isin(VALIDATION_MONTHS)
    ]
    .copy()
    .reset_index(drop=True)
)

validation_df = (
    train_df[
        train_df["mes_validation"].isin(VALIDATION_MONTHS)
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    f"Optimización/train: {optimization_train_df.shape}"
)

print(
    f"Validación:         {validation_df.shape}"
)

print(
    f"Total 2024:         "
    f"{len(optimization_train_df) + len(validation_df)}"
)

Optimización/train: (394560, 28)
Validación:         (132480, 28)
Total 2024:         527040


In [15]:
assert not set(
    optimization_train_df["fecha"]
).intersection(
    set(validation_df["fecha"])
)

assert (
    len(optimization_train_df)
    + len(validation_df)
    == len(train_df)
)

In [16]:
for target in TARGETS:

    print(f"\n{'=' * 70}")
    print(target)
    print(f"{'=' * 70}")

    comparison = pd.concat(
        [
            optimization_train_df[target]
            .value_counts()
            .sort_index()
            .rename("train_count"),

            validation_df[target]
            .value_counts()
            .sort_index()
            .rename("validation_count"),

            optimization_train_df[target]
            .value_counts(normalize=True)
            .sort_index()
            .rename("train_ratio"),

            validation_df[target]
            .value_counts(normalize=True)
            .sort_index()
            .rename("validation_ratio")
        ],
        axis=1
    )

    display(comparison)


codigo_ghi


,train_count,validation_count,train_ratio,validation_ratio
codigo_ghi,,,,
0,331678,105366,0.840628,0.795335
1,62882,27114,0.159372,0.204665



codigo_dni


,train_count,validation_count,train_ratio,validation_ratio
codigo_dni,,,,
0,333635,106275,0.845587,0.802197
1,60224,26063,0.152636,0.196732
2,701,142,0.001777,0.001072



codigo_dhi


,train_count,validation_count,train_ratio,validation_ratio
codigo_dhi,,,,
0,298008,94044,0.755292,0.709873
1,96396,38386,0.244313,0.289749
2,156,50,0.000395,0.000377


In [17]:
for target in TARGETS:

    train_classes = set(
        optimization_train_df[target].unique()
    )

    validation_classes = set(
        validation_df[target].unique()
    )

    print(
        f"{target}: "
        f"train={sorted(train_classes)} | "
        f"validation={sorted(validation_classes)}"
    )

    assert train_classes == validation_classes

codigo_ghi: train=[np.int64(0), np.int64(1)] | validation=[np.int64(0), np.int64(1)]
codigo_dni: train=[np.int64(0), np.int64(1), np.int64(2)] | validation=[np.int64(0), np.int64(1), np.int64(2)]
codigo_dhi: train=[np.int64(0), np.int64(1), np.int64(2)] | validation=[np.int64(0), np.int64(1), np.int64(2)]


In [18]:
from src.preprocessing.model_preprocessing import convert_model_dtypes, prepare_model_features

MODEL_PREPROCESSING = {
    "MLP": {
        "accepts_nan": False,
        "scale": True,
    },
    "LogisticRegression": {
        "accepts_nan": False,
        "scale": True,
    },
    "HistGradientBoosting": {
        "accepts_nan": True,
        "scale": False,
    },
    "HistGradientBoostingClassifier": {
        "accepts_nan": True,
        "scale": False,
    },
}

In [19]:
optimization_train_df = convert_model_dtypes(
    optimization_train_df
)

validation_df = convert_model_dtypes(
    validation_df
)

optimization_train_df.dtypes

ano                              int64
mes_sin                        float64
mes_cos                        float64
dia                              int64
hora_sin                       float64
hora_cos                       float64
minuto                           int64
fecha                   datetime64[us]
ghi                            float64
dni                            float64
dhi                            float64
ghi_estimado                   float64
irr_null                          int8
error_balance                  float64
error_balance_abs              float64
error_balance_rel              float64
elevacion_solar                float64
periodo_solar                     int8
temperatura                    float64
velocidad_viento               float64
humedad_relativa               float64
direccion_viento_sin           float64
direccion_viento_cos           float64
var_meteo_imp                     int8
codigo_ghi                       int64
codigo_dni               

In [20]:
optimization_data = {}

for _, row in candidates_to_optimize.iterrows():

    model_id = int(row["model_id"])
    model_name = row["model_name"]
    target = row["target"]

    features = list(row["features"])

    preprocessing_config = MODEL_PREPROCESSING[
        model_name
    ]

    (
        X_train_opt,
        X_val,
        used_features,
        scaler,
    ) = prepare_model_features(
        train_df=optimization_train_df,
        test_df=validation_df,
        features=features,
        accepts_nan=preprocessing_config["accepts_nan"],
        scale=preprocessing_config["scale"],
    )

    y_train_opt = (
        optimization_train_df[target]
        .copy()
        .reset_index(drop=True)
    )

    y_val = (
        validation_df[target]
        .copy()
        .reset_index(drop=True)
    )

    X_train_opt = X_train_opt.reset_index(drop=True)
    X_val = X_val.reset_index(drop=True)

    optimization_data[model_id] = {
        "model_id": model_id,
        "model_name": model_name,
        "target": target,
        "original_features": features,
        "used_features": used_features,
        "X_train": X_train_opt,
        "X_val": X_val,
        "y_train": y_train_opt,
        "y_val": y_val,
        "scaler": scaler,
        "baseline_f1_2023": row["f1_macro"],
        "baseline_balanced_accuracy_2023": (
            row["balanced_accuracy"]
        ),
        "baseline_f1_weighted_2023": (
            row["f1_weighted"]
        ),
        "original_hyperparameters": (
            row["hyperparameters"]
        ),
    }

In [21]:
preparation_summary = []

for model_id, data in optimization_data.items():

    preparation_summary.append({
        "model_id": model_id,
        "target": data["target"],
        "model_name": data["model_name"],
        "n_original_features": len(
            data["original_features"]
        ),
        "n_used_features": len(
            data["used_features"]
        ),
        "train_samples": len(
            data["X_train"]
        ),
        "validation_samples": len(
            data["X_val"]
        ),
        "train_nan": (
            data["X_train"]
            .isna()
            .sum()
            .sum()
        ),
        "validation_nan": (
            data["X_val"]
            .isna()
            .sum()
            .sum()
        ),
    })

preparation_summary = pd.DataFrame(
    preparation_summary
)

preparation_summary

,model_id,target,model_name,n_original_features,n_used_features,train_samples,validation_samples,train_nan,validation_nan
0,22,codigo_ghi,MLP,14,14,394560,132480,0,0
1,13,codigo_ghi,LogisticRegression,14,14,394560,132480,0,0
2,20,codigo_dni,HistGradientBoosting,18,18,394560,132480,245648,165007
3,3,codigo_dhi,HistGradientBoostingClassifier,22,21,394560,132480,247127,193771


In [22]:
from time import perf_counter

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score
)

In [23]:
OPTIMIZATION_OUTPUT_PATH = Path(
    "../outputs/tables/model_optimization"
)

OPTIMIZATION_OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

HGB_DNI_RESULTS_PATH = (
    OPTIMIZATION_OUTPUT_PATH
    / "hgb_dni_tuning.parquet"
)

HGB_DHI_RESULTS_PATH = (
    OPTIMIZATION_OUTPUT_PATH
    / "hgb_dhi_tuning.parquet"
)

HGB_BEST_RESULTS_PATH = (
    OPTIMIZATION_OUTPUT_PATH
    / "hgb_best_results.parquet"
)

FORCE_RETRAIN_HGB = False

In [24]:
hgb_param_space = {
    "learning_rate": [0.03, 0.05, 0.08, 0.10, 0.15],
    "max_iter": [100, 200, 300, 500],
    "max_leaf_nodes": [15, 31, 63],
    "max_depth": [None, 5, 10, 15],
    "min_samples_leaf": [10, 20, 40, 80],
    "l2_regularization": [0.0, 0.1, 1.0, 5.0, 10.0]
}

In [25]:
def optimize_hgb(
    data,
    param_space,
    n_iter=25,
    random_state=42
):

    X_train = data["X_train"]
    X_val = data["X_val"]
    y_train = data["y_train"]
    y_val = data["y_val"]

    sampled_params = list(
        ParameterSampler(
            param_space,
            n_iter=n_iter,
            random_state=random_state
        )
    )

    results = []

    for i, params in enumerate(sampled_params, start=1):

        start_time = perf_counter()

        model = HistGradientBoostingClassifier(
            **params,
            random_state=random_state
        )

        model.fit(
            X_train,
            y_train
        )

        y_train_pred = model.predict(X_train)
        y_val_pred = model.predict(X_val)

        train_f1_macro = f1_score(
            y_train,
            y_train_pred,
            average="macro",
            zero_division=0
        )

        val_f1_macro = f1_score(
            y_val,
            y_val_pred,
            average="macro",
            zero_division=0
        )

        val_balanced_accuracy = balanced_accuracy_score(
            y_val,
            y_val_pred
        )

        val_f1_weighted = f1_score(
            y_val,
            y_val_pred,
            average="weighted",
            zero_division=0
        )

        elapsed_time = perf_counter() - start_time

        results.append({
            "iteration": i,
            **params,
            "train_f1_macro": train_f1_macro,
            "val_f1_macro": val_f1_macro,
            "overfit_gap": (
                train_f1_macro - val_f1_macro
            ),
            "val_balanced_accuracy": (
                val_balanced_accuracy
            ),
            "val_f1_weighted": (
                val_f1_weighted
            ),
            "training_time_s": elapsed_time
        })

        print(
            f"{i:02d}/{len(sampled_params)} | "
            f"Val F1 macro: {val_f1_macro:.4f} | "
            f"Gap: {train_f1_macro - val_f1_macro:.4f}"
        )

    results_df = pd.DataFrame(results)

    results_df = (
        results_df
        .sort_values(
            by=[
                "val_f1_macro",
                "val_balanced_accuracy",
                "val_f1_weighted"
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )

    return results_df

In [26]:
hgb_candidates = {
    model_id: data
    for model_id, data in optimization_data.items()
    if "HistGradientBoosting" in data["model_name"]
}

for model_id, data in hgb_candidates.items():

    print(
        f"Model ID: {model_id} | "
        f"Target: {data['target']} | "
        f"Modelo: {data['model_name']}"
    )

Model ID: 20 | Target: codigo_dni | Modelo: HistGradientBoosting
Model ID: 3 | Target: codigo_dhi | Modelo: HistGradientBoostingClassifier


In [27]:
dni_hgb_id = next(
    model_id
    for model_id, data in hgb_candidates.items()
    if data["target"] == "codigo_dni"
)

if (
    HGB_DNI_RESULTS_PATH.exists()
    and not FORCE_RETRAIN_HGB
):

    print(
        "Cargando resultados existentes "
        "de HGB para codigo_dni..."
    )

    hgb_dni_results = pd.read_parquet(
        HGB_DNI_RESULTS_PATH
    )

else:

    print(
        "Ejecutando optimización de HGB "
        "para codigo_dni..."
    )

    hgb_dni_results = optimize_hgb(
        data=optimization_data[dni_hgb_id],
        param_space=hgb_param_space,
        n_iter=25,
        random_state=RANDOM_STATE
    )

    hgb_dni_results.to_parquet(
        HGB_DNI_RESULTS_PATH,
        index=False
    )

    print(
        f"Resultados guardados en: "
        f"{HGB_DNI_RESULTS_PATH}"
    )

Cargando resultados existentes de HGB para codigo_dni...


In [28]:
hgb_dni_results.head(10)

,iteration,min_samples_leaf,max_leaf_nodes,max_iter,max_depth,learning_rate,l2_regularization,train_f1_macro,val_f1_macro,overfit_gap,val_balanced_accuracy,val_f1_weighted,training_time_s
0,19,40,31,500,5.0,0.08,0.0,0.776347,0.674742,0.101605,0.675225,0.975786,2.640269
1,4,40,63,300,5.0,0.08,0.0,0.777989,0.674228,0.103761,0.675209,0.975765,2.818369
2,1,10,63,500,5.0,0.15,0.0,0.740043,0.638132,0.101911,0.637514,0.970828,2.253829
3,11,20,15,100,NaN,0.15,0.0,0.851244,0.629897,0.221346,0.659281,0.935947,2.402567
4,13,20,63,300,10.0,0.08,1.0,0.997572,0.623700,0.373871,0.635374,0.952698,26.851511
5,5,40,63,100,NaN,0.10,10.0,0.997506,0.618308,0.379198,0.632490,0.951719,19.263198
6,2,10,31,300,10.0,0.15,5.0,0.997815,0.616665,0.381150,0.630809,0.950180,30.088123
7,9,40,63,300,10.0,0.03,0.0,0.997571,0.610910,0.386661,0.630261,0.939381,49.264796
8,25,40,31,300,5.0,0.15,0.1,0.980315,0.610264,0.370051,0.662346,0.927796,35.152074
9,3,10,63,200,NaN,0.05,5.0,0.998016,0.610105,0.387911,0.628800,0.943281,34.333059


In [29]:
dhi_hgb_id = next(
    model_id
    for model_id, data in hgb_candidates.items()
    if data["target"] == "codigo_dhi"
)

if (
    HGB_DHI_RESULTS_PATH.exists()
    and not FORCE_RETRAIN_HGB
):

    print(
        "Cargando resultados existentes "
        "de HGB para codigo_dhi..."
    )

    hgb_dhi_results = pd.read_parquet(
        HGB_DHI_RESULTS_PATH
    )

else:

    print(
        "Ejecutando optimización de HGB "
        "para codigo_dhi..."
    )

    hgb_dhi_results = optimize_hgb(
        data=optimization_data[dhi_hgb_id],
        param_space=hgb_param_space,
        n_iter=25,
        random_state=RANDOM_STATE
    )

    hgb_dhi_results.to_parquet(
        HGB_DHI_RESULTS_PATH,
        index=False
    )

    print(
        f"Resultados guardados en: "
        f"{HGB_DHI_RESULTS_PATH}"
    )

Cargando resultados existentes de HGB para codigo_dhi...


In [30]:
hgb_dhi_results.head(10)

,iteration,min_samples_leaf,max_leaf_nodes,max_iter,max_depth,learning_rate,l2_regularization,train_f1_macro,val_f1_macro,overfit_gap,val_balanced_accuracy,val_f1_weighted,training_time_s
0,1,10,63,500,5.0,0.15,0.0,0.661159,0.573442,0.087716,0.561731,0.887394,2.224121
1,4,40,63,300,5.0,0.08,0.0,0.772696,0.571140,0.201556,0.554959,0.885785,2.701953
2,19,40,31,500,5.0,0.08,0.0,0.772696,0.571140,0.201556,0.554959,0.885785,2.772486
3,11,20,15,100,NaN,0.15,0.0,0.774794,0.566881,0.207913,0.560479,0.878133,2.548380
4,9,40,63,300,10.0,0.03,0.0,0.774970,0.536424,0.238546,0.548711,0.832777,15.380884
5,3,10,63,200,NaN,0.05,5.0,0.998628,0.531380,0.467247,0.545651,0.826918,35.537108
6,5,40,63,100,NaN,0.10,10.0,0.993091,0.531008,0.462083,0.546052,0.826140,19.081350
7,22,80,31,300,10.0,0.03,1.0,0.998651,0.528822,0.469829,0.544928,0.822834,41.139132
8,23,80,63,100,5.0,0.15,1.0,0.998601,0.527721,0.470879,0.539896,0.822992,14.828369
9,21,40,15,200,5.0,0.10,1.0,0.998155,0.527568,0.470587,0.536928,0.823949,22.702374


In [31]:
best_hgb_results = pd.DataFrame([
    {
        "target": "codigo_dni",
        **hgb_dni_results.iloc[0].to_dict()
    },
    {
        "target": "codigo_dhi",
        **hgb_dhi_results.iloc[0].to_dict()
    }
])

best_hgb_results.to_parquet(
    HGB_BEST_RESULTS_PATH,
    index=False
)

best_hgb_results[
    [
        "target",
        "val_f1_macro",
        "train_f1_macro",
        "overfit_gap",
        "val_balanced_accuracy",
        "val_f1_weighted",
        "learning_rate",
        "max_iter",
        "max_leaf_nodes",
        "max_depth",
        "min_samples_leaf",
        "l2_regularization"
    ]
]

,target,val_f1_macro,train_f1_macro,overfit_gap,val_balanced_accuracy,val_f1_weighted,learning_rate,max_iter,max_leaf_nodes,max_depth,min_samples_leaf,l2_regularization
0,codigo_dni,0.674742,0.776347,0.101605,0.675225,0.975786,0.08,500.0,31.0,5.0,40.0,0.0
1,codigo_dhi,0.573442,0.661159,0.087716,0.561731,0.887394,0.15,500.0,63.0,5.0,10.0,0.0


In [32]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import ParameterSampler

LR_GHI_RESULTS_PATH = (
    OPTIMIZATION_OUTPUT_PATH
    / "lr_ghi_tuning.parquet"
)

FORCE_RETRAIN_LR = False

In [33]:
lr_param_space = [
    {
        "solver": ["lbfgs"],
        "penalty": ["l2"],
        "C": np.logspace(-3, 2, 12),
        "class_weight": [None, "balanced"],
        "max_iter": [1000]
    },
    {
        "solver": ["liblinear"],
        "penalty": ["l1", "l2"],
        "C": np.logspace(-3, 2, 12),
        "class_weight": [None, "balanced"],
        "max_iter": [1000]
    }
]

In [34]:
def optimize_logistic_regression(
    data,
    param_space,
    n_iter=30,
    random_state=42
):

    X_train = data["X_train"]
    X_val = data["X_val"]
    y_train = data["y_train"]
    y_val = data["y_val"]

    sampled_params = list(
        ParameterSampler(
            param_space,
            n_iter=n_iter,
            random_state=random_state
        )
    )

    results = []

    for i, params in enumerate(
        sampled_params,
        start=1
    ):

        start_time = perf_counter()

        model = LogisticRegression(
            **params,
            random_state=random_state
        )

        model.fit(
            X_train,
            y_train
        )

        y_train_pred = model.predict(
            X_train
        )

        y_val_pred = model.predict(
            X_val
        )

        train_f1_macro = f1_score(
            y_train,
            y_train_pred,
            average="macro",
            zero_division=0
        )

        val_f1_macro = f1_score(
            y_val,
            y_val_pred,
            average="macro",
            zero_division=0
        )

        val_balanced_accuracy = (
            balanced_accuracy_score(
                y_val,
                y_val_pred
            )
        )

        val_f1_weighted = f1_score(
            y_val,
            y_val_pred,
            average="weighted",
            zero_division=0
        )

        elapsed_time = (
            perf_counter()
            - start_time
        )

        results.append({
            "iteration": i,
            **params,
            "train_f1_macro": train_f1_macro,
            "val_f1_macro": val_f1_macro,
            "overfit_gap": (
                train_f1_macro
                - val_f1_macro
            ),
            "val_balanced_accuracy": (
                val_balanced_accuracy
            ),
            "val_f1_weighted": (
                val_f1_weighted
            ),
            "training_time_s": (
                elapsed_time
            )
        })

        print(
            f"{i:02d}/{len(sampled_params)} | "
            f"Val F1 macro: "
            f"{val_f1_macro:.4f} | "
            f"Gap: "
            f"{train_f1_macro - val_f1_macro:.4f}"
        )

    results_df = pd.DataFrame(
        results
    )

    results_df = (
        results_df
        .sort_values(
            by=[
                "val_f1_macro",
                "val_balanced_accuracy",
                "val_f1_weighted"
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )

    return results_df

In [35]:
lr_ghi_id = next(
    model_id
    for model_id, data
    in optimization_data.items()
    if (
        data["target"] == "codigo_ghi"
        and data["model_name"]
        == "LogisticRegression"
    )
)

print(
    f"Model ID: {lr_ghi_id} | "
    f"Target: "
    f"{optimization_data[lr_ghi_id]['target']} | "
    f"Modelo: "
    f"{optimization_data[lr_ghi_id]['model_name']}"
)

Model ID: 13 | Target: codigo_ghi | Modelo: LogisticRegression


In [36]:
if (
    LR_GHI_RESULTS_PATH.exists()
    and not FORCE_RETRAIN_LR
):

    print(
        "Cargando resultados existentes "
        "de Logistic Regression "
        "para codigo_ghi..."
    )

    lr_ghi_results = pd.read_parquet(
        LR_GHI_RESULTS_PATH
    )

else:

    print(
        "Ejecutando optimización de "
        "Logistic Regression "
        "para codigo_ghi..."
    )

    lr_ghi_results = (
        optimize_logistic_regression(
            data=optimization_data[
                lr_ghi_id
            ],
            param_space=lr_param_space,
            n_iter=30,
            random_state=RANDOM_STATE
        )
    )

    lr_ghi_results.to_parquet(
        LR_GHI_RESULTS_PATH,
        index=False
    )

    print(
        f"Resultados guardados en: "
        f"{LR_GHI_RESULTS_PATH}"
    )

Cargando resultados existentes de Logistic Regression para codigo_ghi...


In [37]:
lr_ghi_results.head(10)

,iteration,solver,penalty,max_iter,class_weight,C,train_f1_macro,val_f1_macro,overfit_gap,val_balanced_accuracy,val_f1_weighted,training_time_s
0,19,liblinear,l2,1000,NaN,0.008111,0.655541,0.678521,-0.022979,0.642665,0.817458,1.359131
1,7,lbfgs,l2,1000,NaN,0.187382,0.654986,0.677778,-0.022792,0.642218,0.816950,0.773783
2,9,lbfgs,l2,1000,NaN,0.533670,0.654936,0.677767,-0.022832,0.642217,0.816937,0.767323
3,1,lbfgs,l2,1000,NaN,0.008111,0.655695,0.677762,-0.022068,0.642084,0.817069,0.808046
4,16,lbfgs,l2,1000,NaN,100.000000,0.654934,0.677737,-0.022803,0.642190,0.816925,0.764119
5,3,lbfgs,l2,1000,NaN,12.328467,0.654934,0.677728,-0.022794,0.642185,0.816919,0.763198
6,22,lbfgs,l2,1000,NaN,4.328761,0.654934,0.677728,-0.022794,0.642185,0.816919,0.761846
7,14,liblinear,l2,1000,NaN,0.187382,0.654897,0.677700,-0.022802,0.642154,0.816914,1.272729
8,24,liblinear,l1,1000,NaN,0.187382,0.654863,0.677623,-0.022760,0.642085,0.816885,11.694861
9,25,liblinear,l2,1000,NaN,100.000000,0.654833,0.677575,-0.022742,0.642074,0.816833,1.279999


In [38]:
best_lr_ghi = (
    lr_ghi_results
    .iloc[0]
    .copy()
)

display(
    pd.DataFrame(
        [best_lr_ghi]
    )[
        [
            "val_f1_macro",
            "train_f1_macro",
            "overfit_gap",
            "val_balanced_accuracy",
            "val_f1_weighted",
            "C",
            "solver",
            "penalty",
            "class_weight",
            "max_iter",
            "training_time_s"
        ]
    ]
)

,val_f1_macro,train_f1_macro,overfit_gap,val_balanced_accuracy,val_f1_weighted,C,solver,penalty,class_weight,max_iter,training_time_s
0,0.678521,0.655541,-0.022979,0.642665,0.817458,0.008111,liblinear,l2,NaN,1000,1.359131


In [39]:
LR_GHI_BEST_PATH = (
    OPTIMIZATION_OUTPUT_PATH
    / "lr_ghi_best.parquet"
)

best_lr_ghi_df = pd.DataFrame(
    [best_lr_ghi]
)

best_lr_ghi_df.to_parquet(
    LR_GHI_BEST_PATH,
    index=False
)

In [40]:
import json
import tensorflow as tf

from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping

In [41]:
MLP_GHI_RESULTS_PATH = (
    OPTIMIZATION_OUTPUT_PATH
    / "mlp_ghi_tuning.parquet"
)

MLP_GHI_BEST_PATH = (
    OPTIMIZATION_OUTPUT_PATH
    / "mlp_ghi_best.parquet"
)

FORCE_RETRAIN_MLP = False

In [42]:
mlp_ghi_id = next(
    model_id
    for model_id, data in optimization_data.items()
    if (
        data["target"] == "codigo_ghi"
        and data["model_name"] == "MLP"
    )
)

print(
    f"Model ID: {mlp_ghi_id} | "
    f"Target: {optimization_data[mlp_ghi_id]['target']} | "
    f"Modelo: {optimization_data[mlp_ghi_id]['model_name']}"
)

Model ID: 22 | Target: codigo_ghi | Modelo: MLP


In [43]:
def build_mlp(
    input_dim,
    hidden_units,
    dropout_rate,
    l2_strength,
    learning_rate,
    random_state=42
):

    tf.keras.utils.set_random_seed(
        random_state
    )

    model = Sequential()

    model.add(
        Input(shape=(input_dim,))
    )

    for units in hidden_units:

        model.add(
            Dense(
                units,
                activation="relu",
                kernel_regularizer=l2(
                    l2_strength
                )
            )
        )

        if dropout_rate > 0:
            model.add(
                Dropout(dropout_rate)
            )

    model.add(
        Dense(
            1,
            activation="sigmoid"
        )
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="binary_crossentropy"
    )

    return model

In [44]:
mlp_param_space = {
    "hidden_units": [
        (64,),
        (128,),
        (64, 32),
        (128, 64),
        (128, 64, 32),
        (256, 128, 64)
    ],
    "dropout_rate": [
        0.1,
        0.2,
        0.3,
        0.4,
        0.5
    ],
    "l2_strength": [
        0.0,
        1e-5,
        1e-4,
        1e-3
    ],
    "learning_rate": [
        1e-4,
        3e-4,
        1e-3
    ],
    "batch_size": [
        512,
        1024,
        2048
    ],
    "class_weight_mode": [
        "none",
        "balanced"
    ]
}

In [45]:
def optimize_mlp(
    data,
    param_space,
    n_iter=15,
    max_epochs=100,
    patience=8,
    random_state=42
):

    X_train = (
        data["X_train"]
        .to_numpy(dtype=np.float32)
    )

    X_val = (
        data["X_val"]
        .to_numpy(dtype=np.float32)
    )

    y_train = (
        data["y_train"]
        .to_numpy(dtype=np.int32)
    )

    y_val = (
        data["y_val"]
        .to_numpy(dtype=np.int32)
    )

    sampled_params = list(
        ParameterSampler(
            param_space,
            n_iter=n_iter,
            random_state=random_state
        )
    )

    classes = np.unique(y_train)

    balanced_weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )

    balanced_class_weight = dict(
        zip(
            classes,
            balanced_weights
        )
    )

    results = []

    for i, params in enumerate(
        sampled_params,
        start=1
    ):

        tf.keras.backend.clear_session()

        start_time = perf_counter()

        model = build_mlp(
            input_dim=X_train.shape[1],
            hidden_units=params[
                "hidden_units"
            ],
            dropout_rate=params[
                "dropout_rate"
            ],
            l2_strength=params[
                "l2_strength"
            ],
            learning_rate=params[
                "learning_rate"
            ],
            random_state=random_state
        )

        if (
            params["class_weight_mode"]
            == "balanced"
        ):
            class_weight = (
                balanced_class_weight
            )
        else:
            class_weight = None

        early_stopping = EarlyStopping(
            monitor="val_loss",
            patience=patience,
            restore_best_weights=True,
            mode="min"
        )

        history = model.fit(
            X_train,
            y_train,
            validation_data=(
                X_val,
                y_val
            ),
            epochs=max_epochs,
            batch_size=params[
                "batch_size"
            ],
            class_weight=class_weight,
            callbacks=[
                early_stopping
            ],
            verbose=0
        )

        train_prob = model.predict(
            X_train,
            batch_size=4096,
            verbose=0
        ).ravel()

        val_prob = model.predict(
            X_val,
            batch_size=4096,
            verbose=0
        ).ravel()

        train_pred = (
            train_prob >= 0.5
        ).astype(int)

        val_pred = (
            val_prob >= 0.5
        ).astype(int)

        train_f1_macro = f1_score(
            y_train,
            train_pred,
            average="macro",
            zero_division=0
        )

        val_f1_macro = f1_score(
            y_val,
            val_pred,
            average="macro",
            zero_division=0
        )

        val_balanced_accuracy = (
            balanced_accuracy_score(
                y_val,
                val_pred
            )
        )

        val_f1_weighted = f1_score(
            y_val,
            val_pred,
            average="weighted",
            zero_division=0
        )

        val_losses = history.history[
            "val_loss"
        ]

        best_epoch = (
            int(np.argmin(val_losses))
            + 1
        )

        elapsed_time = (
            perf_counter()
            - start_time
        )

        results.append({
            "iteration": i,
            "hidden_units": json.dumps(
                list(
                    params[
                        "hidden_units"
                    ]
                )
            ),
            "dropout_rate": params[
                "dropout_rate"
            ],
            "l2_strength": params[
                "l2_strength"
            ],
            "learning_rate": params[
                "learning_rate"
            ],
            "batch_size": params[
                "batch_size"
            ],
            "class_weight_mode": params[
                "class_weight_mode"
            ],
            "epochs_trained": len(
                history.history["loss"]
            ),
            "best_epoch": best_epoch,
            "best_val_loss": float(
                np.min(val_losses)
            ),
            "train_f1_macro": (
                train_f1_macro
            ),
            "val_f1_macro": (
                val_f1_macro
            ),
            "overfit_gap": (
                train_f1_macro
                - val_f1_macro
            ),
            "val_balanced_accuracy": (
                val_balanced_accuracy
            ),
            "val_f1_weighted": (
                val_f1_weighted
            ),
            "training_time_s": (
                elapsed_time
            )
        })

        print(
            f"{i:02d}/{len(sampled_params)} | "
            f"Val F1: {val_f1_macro:.4f} | "
            f"Train F1: {train_f1_macro:.4f} | "
            f"Gap: "
            f"{train_f1_macro - val_f1_macro:.4f} | "
            f"Epoch: {best_epoch}"
        )

    results_df = pd.DataFrame(
        results
    )

    results_df = (
        results_df
        .sort_values(
            by=[
                "val_f1_macro",
                "overfit_gap",
                "val_balanced_accuracy"
            ],
            ascending=[
                False,
                True,
                False
            ]
        )
        .reset_index(drop=True)
    )

    return results_df

In [46]:
def optimize_mlp(
    data,
    param_space,
    n_iter=15,
    max_epochs=100,
    patience=8,
    random_state=42
):

    X_train = (
        data["X_train"]
        .to_numpy(dtype=np.float32)
    )

    X_val = (
        data["X_val"]
        .to_numpy(dtype=np.float32)
    )

    y_train = (
        data["y_train"]
        .to_numpy(dtype=np.int32)
    )

    y_val = (
        data["y_val"]
        .to_numpy(dtype=np.int32)
    )

    sampled_params = list(
        ParameterSampler(
            param_space,
            n_iter=n_iter,
            random_state=random_state
        )
    )

    classes = np.unique(y_train)

    balanced_weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )

    balanced_class_weight = dict(
        zip(
            classes,
            balanced_weights
        )
    )

    results = []

    for i, params in enumerate(
        sampled_params,
        start=1
    ):

        tf.keras.backend.clear_session()

        start_time = perf_counter()

        model = build_mlp(
            input_dim=X_train.shape[1],
            hidden_units=params[
                "hidden_units"
            ],
            dropout_rate=params[
                "dropout_rate"
            ],
            l2_strength=params[
                "l2_strength"
            ],
            learning_rate=params[
                "learning_rate"
            ],
            random_state=random_state
        )

        if (
            params["class_weight_mode"]
            == "balanced"
        ):
            class_weight = (
                balanced_class_weight
            )
        else:
            class_weight = None

        early_stopping = EarlyStopping(
            monitor="val_loss",
            patience=patience,
            restore_best_weights=True,
            mode="min"
        )

        history = model.fit(
            X_train,
            y_train,
            validation_data=(
                X_val,
                y_val
            ),
            epochs=max_epochs,
            batch_size=params[
                "batch_size"
            ],
            class_weight=class_weight,
            callbacks=[
                early_stopping
            ],
            verbose=0
        )

        train_prob = model.predict(
            X_train,
            batch_size=4096,
            verbose=0
        ).ravel()

        val_prob = model.predict(
            X_val,
            batch_size=4096,
            verbose=0
        ).ravel()

        train_pred = (
            train_prob >= 0.5
        ).astype(int)

        val_pred = (
            val_prob >= 0.5
        ).astype(int)

        train_f1_macro = f1_score(
            y_train,
            train_pred,
            average="macro",
            zero_division=0
        )

        val_f1_macro = f1_score(
            y_val,
            val_pred,
            average="macro",
            zero_division=0
        )

        val_balanced_accuracy = (
            balanced_accuracy_score(
                y_val,
                val_pred
            )
        )

        val_f1_weighted = f1_score(
            y_val,
            val_pred,
            average="weighted",
            zero_division=0
        )

        val_losses = history.history[
            "val_loss"
        ]

        best_epoch = (
            int(np.argmin(val_losses))
            + 1
        )

        elapsed_time = (
            perf_counter()
            - start_time
        )

        results.append({
            "iteration": i,
            "hidden_units": json.dumps(
                list(
                    params[
                        "hidden_units"
                    ]
                )
            ),
            "dropout_rate": params[
                "dropout_rate"
            ],
            "l2_strength": params[
                "l2_strength"
            ],
            "learning_rate": params[
                "learning_rate"
            ],
            "batch_size": params[
                "batch_size"
            ],
            "class_weight_mode": params[
                "class_weight_mode"
            ],
            "epochs_trained": len(
                history.history["loss"]
            ),
            "best_epoch": best_epoch,
            "best_val_loss": float(
                np.min(val_losses)
            ),
            "train_f1_macro": (
                train_f1_macro
            ),
            "val_f1_macro": (
                val_f1_macro
            ),
            "overfit_gap": (
                train_f1_macro
                - val_f1_macro
            ),
            "val_balanced_accuracy": (
                val_balanced_accuracy
            ),
            "val_f1_weighted": (
                val_f1_weighted
            ),
            "training_time_s": (
                elapsed_time
            )
        })

        print(
            f"{i:02d}/{len(sampled_params)} | "
            f"Val F1: {val_f1_macro:.4f} | "
            f"Train F1: {train_f1_macro:.4f} | "
            f"Gap: "
            f"{train_f1_macro - val_f1_macro:.4f} | "
            f"Epoch: {best_epoch}"
        )

    results_df = pd.DataFrame(
        results
    )

    results_df = (
        results_df
        .sort_values(
            by=[
                "val_f1_macro",
                "overfit_gap",
                "val_balanced_accuracy"
            ],
            ascending=[
                False,
                True,
                False
            ]
        )
        .reset_index(drop=True)
    )

    return results_df

In [48]:
if (
    MLP_GHI_RESULTS_PATH.exists()
    and not FORCE_RETRAIN_MLP
):

    print(
        "Cargando resultados existentes "
        "del MLP para codigo_ghi..."
    )

    mlp_ghi_results = pd.read_parquet(
        MLP_GHI_RESULTS_PATH
    )

else:

    print(
        "Ejecutando optimización del "
        "MLP para codigo_ghi..."
    )

    mlp_ghi_results = optimize_mlp(
        data=optimization_data[
            mlp_ghi_id
        ],
        param_space=mlp_param_space,
        n_iter=15,
        max_epochs=100,
        patience=8,
        random_state=RANDOM_STATE
    )

    mlp_ghi_results.to_parquet(
        MLP_GHI_RESULTS_PATH,
        index=False
    )

    print(
        f"Resultados guardados en: "
        f"{MLP_GHI_RESULTS_PATH}"
    )

Cargando resultados existentes del MLP para codigo_ghi...


In [49]:
mlp_ghi_results.head(10)

,iteration,hidden_units,dropout_rate,l2_strength,learning_rate,batch_size,class_weight_mode,epochs_trained,best_epoch,best_val_loss,train_f1_macro,val_f1_macro,overfit_gap,val_balanced_accuracy,val_f1_weighted,training_time_s
0,5,"[128, 64, 32]",0.3,0.00010,0.0001,2048,none,14,6,0.398113,0.732697,0.777925,-0.045229,0.732300,0.868106,17.892930
1,9,"[128, 64]",0.1,0.00010,0.0001,2048,none,12,4,0.414235,0.693923,0.748232,-0.054309,0.703711,0.852291,12.013851
2,8,"[128, 64]",0.5,0.00010,0.0001,512,none,11,3,0.407631,0.753644,0.748042,0.005602,0.711284,0.849104,25.018078
3,13,"[128, 64, 32]",0.1,0.00000,0.0003,1024,none,9,1,0.402295,0.787989,0.742822,0.045166,0.718306,0.841473,15.385577
4,14,[64],0.2,0.00001,0.0001,2048,none,46,38,0.386761,0.827501,0.731197,0.096303,0.722207,0.828458,26.784552
5,12,"[64, 32]",0.4,0.00001,0.0010,2048,none,9,1,0.412693,0.684144,0.705659,-0.021516,0.669143,0.828078,7.937252
6,11,"[128, 64, 32]",0.2,0.00100,0.0003,512,none,9,1,0.500616,0.820632,0.686293,0.134339,0.685839,0.795991,22.943147
7,1,"[256, 128, 64]",0.2,0.00010,0.0010,1024,none,9,1,0.798476,0.920676,0.608474,0.312203,0.634477,0.721722,22.533111
8,4,[128],0.1,0.00001,0.0001,1024,balanced,23,15,0.606006,0.773420,0.607702,0.165719,0.687661,0.689748,27.739469
9,6,"[64, 32]",0.2,0.00100,0.0003,512,balanced,9,1,0.652098,0.725677,0.605979,0.119698,0.652338,0.706150,19.551040


In [50]:
display(
    mlp_ghi_results[
        [
            "val_f1_macro",
            "train_f1_macro",
            "overfit_gap",
            "val_balanced_accuracy",
            "val_f1_weighted",
            "hidden_units",
            "dropout_rate",
            "l2_strength",
            "learning_rate",
            "batch_size",
            "class_weight_mode",
            "best_epoch",
            "epochs_trained"
        ]
    ].head(10)
)

,val_f1_macro,train_f1_macro,overfit_gap,val_balanced_accuracy,val_f1_weighted,hidden_units,dropout_rate,l2_strength,learning_rate,batch_size,class_weight_mode,best_epoch,epochs_trained
0,0.777925,0.732697,-0.045229,0.732300,0.868106,"[128, 64, 32]",0.3,0.00010,0.0001,2048,none,6,14
1,0.748232,0.693923,-0.054309,0.703711,0.852291,"[128, 64]",0.1,0.00010,0.0001,2048,none,4,12
2,0.748042,0.753644,0.005602,0.711284,0.849104,"[128, 64]",0.5,0.00010,0.0001,512,none,3,11
3,0.742822,0.787989,0.045166,0.718306,0.841473,"[128, 64, 32]",0.1,0.00000,0.0003,1024,none,1,9
4,0.731197,0.827501,0.096303,0.722207,0.828458,[64],0.2,0.00001,0.0001,2048,none,38,46
5,0.705659,0.684144,-0.021516,0.669143,0.828078,"[64, 32]",0.4,0.00001,0.0010,2048,none,1,9
6,0.686293,0.820632,0.134339,0.685839,0.795991,"[128, 64, 32]",0.2,0.00100,0.0003,512,none,1,9
7,0.608474,0.920676,0.312203,0.634477,0.721722,"[256, 128, 64]",0.2,0.00010,0.0010,1024,none,1,9
8,0.607702,0.773420,0.165719,0.687661,0.689748,[128],0.1,0.00001,0.0001,1024,balanced,15,23
9,0.605979,0.725677,0.119698,0.652338,0.706150,"[64, 32]",0.2,0.00100,0.0003,512,balanced,1,9


In [51]:
best_mlp_ghi = (
    mlp_ghi_results
    .iloc[0]
    .copy()
)

best_mlp_ghi_df = pd.DataFrame(
    [best_mlp_ghi]
)

best_mlp_ghi_df.to_parquet(
    MLP_GHI_BEST_PATH,
    index=False
)

In [52]:
best_mlp_ghi = (
    mlp_ghi_results
    .iloc[0]
    .copy()
)

best_mlp_ghi_df = pd.DataFrame(
    [best_mlp_ghi]
)

best_mlp_ghi_df.to_parquet(
    MLP_GHI_BEST_PATH,
    index=False
)

In [53]:
ghi_model_comparison = pd.DataFrame([
    {
        "model": "LogisticRegression",
        "val_f1_macro": best_lr_ghi["val_f1_macro"],
        "train_f1_macro": best_lr_ghi["train_f1_macro"],
        "overfit_gap": best_lr_ghi["overfit_gap"],
        "val_balanced_accuracy": best_lr_ghi[
            "val_balanced_accuracy"
        ],
        "val_f1_weighted": best_lr_ghi[
            "val_f1_weighted"
        ],
    },
    {
        "model": "MLP",
        "val_f1_macro": best_mlp_ghi["val_f1_macro"],
        "train_f1_macro": best_mlp_ghi["train_f1_macro"],
        "overfit_gap": best_mlp_ghi["overfit_gap"],
        "val_balanced_accuracy": best_mlp_ghi[
            "val_balanced_accuracy"
        ],
        "val_f1_weighted": best_mlp_ghi[
            "val_f1_weighted"
        ],
    }
])

ghi_model_comparison = (
    ghi_model_comparison
    .sort_values(
        by=[
            "val_f1_macro",
            "overfit_gap"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(drop=True)
)

In [54]:
GHI_MODEL_COMPARISON_PATH = (
    OPTIMIZATION_OUTPUT_PATH
    / "ghi_model_comparison.parquet"
)

ghi_model_comparison.to_parquet(
    GHI_MODEL_COMPARISON_PATH,
    index=False
)

ghi_model_comparison.round(4)

,model,val_f1_macro,train_f1_macro,overfit_gap,val_balanced_accuracy,val_f1_weighted
0,MLP,0.7779,0.7327,-0.0452,0.7323,0.8681
1,LogisticRegression,0.6785,0.6555,-0.0230,0.6427,0.8175


In [60]:
final_model_selection = pd.DataFrame([
    {
        "target": "codigo_ghi",
        "model": "MLP",
        "val_f1_macro": best_mlp_ghi["val_f1_macro"],
        "train_f1_macro": best_mlp_ghi["train_f1_macro"],
        "overfit_gap": best_mlp_ghi["overfit_gap"],
        "val_balanced_accuracy": best_mlp_ghi[
            "val_balanced_accuracy"
        ],
        "val_f1_weighted": best_mlp_ghi[
            "val_f1_weighted"
        ],
        "source_model_id": mlp_ghi_id,
    },

    {
        "target": "codigo_dni",
        "model": "HistGradientBoostingClassifier",
        "val_f1_macro": hgb_dni_results.iloc[0][
            "val_f1_macro"
        ],
        "train_f1_macro": hgb_dni_results.iloc[0][
            "train_f1_macro"
        ],
        "overfit_gap": hgb_dni_results.iloc[0][
            "overfit_gap"
        ],
        "val_balanced_accuracy": hgb_dni_results.iloc[0][
            "val_balanced_accuracy"
        ],
        "val_f1_weighted": hgb_dni_results.iloc[0][
            "val_f1_weighted"
        ],
        "source_model_id": dni_hgb_id,
    },

    {
        "target": "codigo_dhi",
        "model": "HistGradientBoostingClassifier",
        "val_f1_macro": hgb_dhi_results.iloc[0][
            "val_f1_macro"
        ],
        "train_f1_macro": hgb_dhi_results.iloc[0][
            "train_f1_macro"
        ],
        "overfit_gap": hgb_dhi_results.iloc[0][
            "overfit_gap"
        ],
        "val_balanced_accuracy": hgb_dhi_results.iloc[0][
            "val_balanced_accuracy"
        ],
        "val_f1_weighted": hgb_dhi_results.iloc[0][
            "val_f1_weighted"
        ],
        "source_model_id": dhi_hgb_id,
    },
])

display(
    final_model_selection.round(4)
)

,target,model,val_f1_macro,train_f1_macro,overfit_gap,val_balanced_accuracy,val_f1_weighted,source_model_id
0,codigo_ghi,MLP,0.7779,0.7327,-0.0452,0.7323,0.8681,22
1,codigo_dni,HistGradientBoostingClassifier,0.6747,0.7763,0.1016,0.6752,0.9758,20
2,codigo_dhi,HistGradientBoostingClassifier,0.5734,0.6612,0.0877,0.5617,0.8874,3
